# Clase 6 parte 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

**Autor: Jaime Andrés Jaramillo Ramírez.**

Ejecutar las partes 1, 2 y 3 en ese orden, desde la misma carpeta (o la subcarpeta `notebooks`). Dependencias: numpy, pandas, matplotlib, seaborn, scikit-learn y joblib. En Colab, usar el mismo entorno de ejecución para conservar los archivos entre partes.

**Fuente:** no se adjuntó el CSV de clase. La parte 1 usa `data/raw/wine.csv` si existe; de lo contrario exporta el dataset Wine incluido en scikit-learn. Los resultados guardados corresponden a esta alternativa y deben recalcularse si se usa otro CSV.

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]
RANDOM_STATE = contract["random_state"]
preprocessor_tree = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), feature_columns)
], remainder="drop")
preprocessor_scaled = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)
], remainder="drop")


**¿Por qué el escalador debe estar dentro del pipeline?**

La media y la desviación estándar se deben estimar únicamente con el entrenamiento de cada fold. Si se ajusta el escalador antes de la validación cruzada, utiliza información de las observaciones que después actuarán como validación: se produce fuga de información y la estimación del desempeño puede ser optimista. El pipeline reajusta imputación y escalado en cada fold y aplica esas mismas transformaciones a su validación. Al seleccionar el campeón, se reajusta el pipeline con todo el entrenamiento. El árbol no necesita escalado porque sus divisiones dependen del orden de los valores.


## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("preprocess", clone(preprocessor_tree)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("preprocess", clone(preprocessor_scaled)),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("preprocess", clone(preprocessor_scaled)),
        ("model", KNeighborsClassifier()),
    ]),
}


## 4. Comparación inicial con validación cruzada

In [6]:
if y_train.value_counts().min() < 5:
    raise ValueError("Se necesitan al menos 5 registros de cada clase en train.")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv,
                            scoring="accuracy", return_train_score=True, n_jobs=1)
    rows.append({"model": name,
                 "train_accuracy": scores["train_score"].mean(),
                 "validation_accuracy": scores["test_score"].mean(),
                 "validation_std": scores["test_score"].std(ddof=0)})
initial_comparison = pd.DataFrame(rows).sort_values("validation_accuracy", ascending=False)
initial_comparison["gap_train_validation"] = initial_comparison["train_accuracy"] - initial_comparison["validation_accuracy"]
display(initial_comparison)


,model,train_accuracy,validation_accuracy,validation_std,gap_train_validation
1,LogisticRegression,1.0000,0.979064,0.027713,0.020936
2,KNN,0.9771,0.950739,0.035274,0.026361
0,DecisionTree,1.0000,0.902217,0.058617,0.097783


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {"model__max_depth": [2, 3, 5, None], "model__min_samples_leaf": [1, 3, 5]},
    "LogisticRegression": {"model__C": [0.01, 0.1, 1, 10]},
    "KNN": {"model__n_neighbors": [3, 5, 7, 9], "model__weights": ["uniform", "distance"]},
}
searches, rows = {}, []
for name, pipeline in models.items():
    search = GridSearchCV(pipeline, param_grids[name], scoring="accuracy", cv=cv,
                          refit=True, return_train_score=True, n_jobs=1, error_score="raise")
    search.fit(X_train, y_train)
    searches[name] = search
    i = search.best_index_
    rows.append({"model": name, "cv_accuracy": search.best_score_,
                 "cv_std": search.cv_results_["std_test_score"][i],
                 "train_accuracy": search.cv_results_["mean_train_score"][i],
                 "best_params": search.best_params_})
tuned_comparison = pd.DataFrame(rows).sort_values("cv_accuracy", ascending=False, kind="stable")
tuned_comparison["gap_train_validation"] = tuned_comparison["train_accuracy"] - tuned_comparison["cv_accuracy"]
display(tuned_comparison)


,model,cv_accuracy,cv_std,train_accuracy,best_params,gap_train_validation
1,LogisticRegression,0.979064,0.027713,1.000000,{'model__C': 1},0.020936
2,KNN,0.957882,0.040520,0.964773,"{'model__n_neighbors': 7, 'model__weights': 'u...",0.006891
0,DecisionTree,0.902217,0.058617,1.000000,"{'model__max_depth': 5, 'model__min_samples_le...",0.097783


## 6. Selección y persistencia del campeón

In [8]:
import sklearn
# En empate exacto se prefiere logística por su sencillez de inferencia;
# después KNN y árbol. Esta regla se fija antes de abrir test.
priority = {"LogisticRegression": 0, "KNN": 1, "DecisionTree": 2}
champion_name = sorted(searches, key=lambda n: (-searches[n].best_score_, priority[n]))[0]
search = searches[champion_name]
champion = search.best_estimator_  # refit=True: ya ajustado con todo train.
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")
i = search.best_index_
metadata = {
    "champion": champion_name, "cv_accuracy": float(search.best_score_),
    "cv_std": float(search.cv_results_["std_test_score"][i]),
    "train_accuracy": float(search.cv_results_["mean_train_score"][i]),
    "best_params": search.best_params_, "random_state": RANDOM_STATE,
    "n_splits": 5, "scoring": "accuracy", "n_train": len(X_train),
    "features": feature_columns, "classes": champion.classes_.tolist(),
    "sklearn_version": sklearn.__version__,
    "selection_rule": "max CV accuracy; exact ties: LogisticRegression, KNN, DecisionTree",
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
all_results = pd.concat([
    pd.DataFrame(s.cv_results_).assign(model=name) for name, s in searches.items()
], ignore_index=True)
all_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
initial_comparison.to_csv(REPORTS_DIR / "initial_comparison.csv", index=False)
tuned_comparison.to_csv(REPORTS_DIR / "tuned_comparison.csv", index=False)
print("Campeón:", champion_name)
print("Accuracy CV:", metadata["cv_accuracy"], "+/-", metadata["cv_std"])
print("Parámetros:", metadata["best_params"])


Campeón: LogisticRegression
Accuracy CV: 0.9790640394088669 +/- 0.027713464392187602
Parámetros: {'model__C': 1}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

Se seleccionó regresión logística con `C=1` por obtener la mayor accuracy media de CV: **97,91 %**, frente a **95,79 %** de KNN y **90,22 %** del árbol ajustado. Su desviación entre folds fue de **2,77 puntos porcentuales**, menor que la de KNN (4,05) y la del árbol (5,86). Esta desviación describe la variabilidad entre folds; no es un intervalo de confianza.

La accuracy media de entrenamiento de logística fue 100 %, con una brecha de 2,09 puntos frente a validación. El árbol presentó una brecha mayor (9,78 puntos), compatible con más sobreajuste. KNN tuvo una brecha menor, pero también un desempeño de validación inferior: una brecha pequeña por sí sola no determina el mejor modelo.

La regresión logística permite examinar coeficientes por clase en la escala estandarizada y tiene inferencia sencilla. Sus coeficientes deben interpretarse con cautela cuando hay predictores correlacionados. Un árbol pequeño ofrece reglas más directas, pero aquí su desempeño fue inferior; KNN depende de las distancias a las observaciones de entrenamiento y es menos resumible mediante parámetros.

La selección utilizó exclusivamente entrenamiento y CV. Como se eligieron hiperparámetros con los mismos folds, el mejor valor de CV puede ser optimista por selección; la evaluación independiente queda reservada a la parte 3. Se guarda el pipeline completo para repetir exactamente el preprocesamiento durante inferencia.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.
